# Notebook 21 - Array, Map, Struct and JSON Extras

**Datasets:** `samples.bakehouse.sales_transactions`, `samples.bakehouse.sales_customers`, `samples.wanderbricks.bookings`  
**Difficulty:** Medium  
**Topics:** `explode_outer`, `posexplode_outer`, `array_remove`, `array_position`, `array_repeat`, `arrays_overlap`, `arrays_zip`, `map_entries`, `map_filter`, `map_concat`, `str_to_map`, `named_struct`, `json_tuple`, `sort_array`, `array_distinct`, `array_union`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_bookings     = spark.table("samples.wanderbricks.bookings")

## Problem 1 - explode_outer and posexplode_outer

Create the following DataFrame:
```python
spark.createDataFrame([(1, ["a","b","c"]), (2, None), (3, ["x"])], ["id", "tags"])
```
Show the difference between `F.explode` (drops null-array rows) and `F.explode_outer` (keeps the null-array row, emitting `null` for the value).  
Also apply `F.posexplode_outer` which yields a position column alongside the value.

**Expected for explode_outer:** the row where `id=2` appears with `tag=null`  
**Expected columns from posexplode_outer:** `id`, `pos`, `tag`  
Store results as `result_1a` (explode), `result_1b` (explode_outer), `result_1c` (posexplode_outer).

In [ ]:
result_1a = None
result_1b = None
result_1c = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 1 --
assert result_1a is not None, "result_1a is None"
assert result_1b is not None, "result_1b is None"
assert result_1c is not None, "result_1c is None"

cnt_a = result_1a.count()
cnt_b = result_1b.count()
assert cnt_b > cnt_a, "explode_outer should produce more rows than explode (null array row included)"

null_rows = result_1b.filter(F.col("tag").isNull())
assert null_rows.count() == 1, "explode_outer should have exactly 1 null-tag row (id=2)"

cols_1c = [c.lower() for c in result_1c.columns]
assert "pos" in cols_1c, "posexplode_outer result must have a 'pos' column"
assert "tag" in cols_1c, "posexplode_outer result must have a 'tag' column"

print(f"Problem 1 passed -  explode rows={cnt_a}, explode_outer rows={cnt_b}")

## Problem 2 - array_remove, array_position, array_repeat

Create the following DataFrame:
```python
spark.createDataFrame([(1,[1,2,3,2,1]),(2,[5,5,5])], ["id","nums"])
```
Apply these three functions and add their results as new columns:
- `F.array_remove(col, 2)` - removes all occurrences of the value `2`
- `F.array_position(col, 5)` - returns the 1-based position of the first occurrence of `5` (`0` if not found)
- `F.array_repeat(F.lit("x"), 3)` - creates the array `["x","x","x"]`

**Expected columns:** `id`, `nums`, `removed_2`, `pos_of_5`, `repeated`  
Store result as `result_2`.

In [ ]:
result_2 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 2 --
assert result_2 is not None, "result_2 is None"

cols_2 = [c.lower() for c in result_2.columns]
for col_name in ["id", "nums", "removed_2", "pos_of_5", "repeated"]:
    assert col_name in cols_2, f"Missing column: {col_name}"

rows = {r["id"]: r for r in result_2.collect()}

# array_remove: row 1 should have no 2s in removed_2
assert 2 not in rows[1]["removed_2"], "array_remove should have removed all 2s from row 1"

# array_position: row 2 has [5,5,5], first 5 is at position 1
assert rows[2]["pos_of_5"] == 1, "array_position of 5 in [5,5,5] should be 1"

# array_position: row 1 has no 5, should return 0
assert rows[1]["pos_of_5"] == 0, "array_position of 5 in [1,2,3,2,1] should be 0"

print("Problem 2 passed -  array_remove, array_position, array_repeat verified")

## Problem 3 - arrays_overlap and arrays_zip

Create the following DataFrame:
```python
spark.createDataFrame([(1,[1,2,3],[3,4,5]),(2,[1,2],[6,7])], ["id","a","b"])
```
Apply:
- `F.arrays_overlap(col_a, col_b)` - returns `true` if the two arrays share at least one element
- `F.arrays_zip(col_a, col_b)` - zips the two arrays element-wise into an array of structs

**Expected columns:** `id`, `a`, `b`, `has_overlap`, `zipped`  
Row 1 shares the element `3`, so `has_overlap=True`. Row 2 has no shared elements, so `has_overlap=False`.  
Store result as `result_3`.

In [ ]:
result_3 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 3 --
assert result_3 is not None, "result_3 is None"

cols_3 = [c.lower() for c in result_3.columns]
for col_name in ["id", "a", "b", "has_overlap", "zipped"]:
    assert col_name in cols_3, f"Missing column: {col_name}"

rows = {r["id"]: r for r in result_3.collect()}
assert rows[1]["has_overlap"] == True, "Row 1 shares element 3 - has_overlap should be True"
assert rows[2]["has_overlap"] == False, "Row 2 has no shared elements - has_overlap should be False"

# zipped should be an ArrayType of StructType
zipped_type = result_3.schema["zipped"].dataType
assert isinstance(zipped_type, T.ArrayType), "zipped must be ArrayType"
assert isinstance(zipped_type.elementType, T.StructType), "zipped elements must be StructType"

print("Problem 3 passed -  arrays_overlap and arrays_zip verified")

## Problem 4 - map_entries, map_filter, map_concat

From `df_transactions`, build a map column per row using `F.create_map` with `product` as the key and `totalPrice` as the value.

Then apply:
- `F.map_entries(map_col)` - converts the map into an array of `(key, value)` structs
- `F.map_filter(map_col, lambda k, v: v > 10)` - keeps only entries where the value exceeds 10
- `F.map_concat(map1, map2)` - merges two maps; create a second map (e.g. `{"source": F.lit("bakehouse")}`) and concatenate

**Expected columns:** `transactionID`, `product_price_map`, `map_entries_col`, `filtered_map`, `concat_map`  
Store result as `result_4`.

In [ ]:
result_4 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 4 --
assert result_4 is not None, "result_4 is None"

cols_4 = [c.lower() for c in result_4.columns]
for col_name in ["transactionid", "product_price_map", "map_entries_col", "filtered_map", "concat_map"]:
    assert col_name in cols_4, f"Missing column: {col_name}"

entries_type = result_4.schema["map_entries_col"].dataType
assert isinstance(entries_type, T.ArrayType), "map_entries_col must be ArrayType"

filtered_type = result_4.schema["filtered_map"].dataType
assert isinstance(filtered_type, T.MapType), "filtered_map must be MapType"

concat_type = result_4.schema["concat_map"].dataType
assert isinstance(concat_type, T.MapType), "concat_map must be MapType"

cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 4 passed -  map_entries, map_filter, map_concat verified ({cnt} rows)")

## Problem 5 - str_to_map

Create the following DataFrame of key-value strings:
```python
spark.createDataFrame([("a=1,b=2,c=3",), ("x=10,y=20",)], ["kv_string"])
```
Use `F.str_to_map(col, pairDelim=",", keyValueDelim="=")` to parse each string into a `MapType(StringType, StringType)` column.  
Then use `F.element_at` to extract the value for key `"a"` into a separate column.

**Expected columns:** `kv_string`, `kv_map`, `val_a`  
For the first row, `val_a` should equal `"1"`.  
Store result as `result_5`.

In [ ]:
result_5 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 5 --
assert result_5 is not None, "result_5 is None"

cols_5 = [c.lower() for c in result_5.columns]
for col_name in ["kv_string", "kv_map", "val_a"]:
    assert col_name in cols_5, f"Missing column: {col_name}"

map_type = result_5.schema["kv_map"].dataType
assert isinstance(map_type, T.MapType), "kv_map must be MapType"

rows = result_5.collect()
row_a = next(r for r in rows if "a=1" in r["kv_string"])
assert row_a["val_a"] == "1", f"val_a for 'a=1,b=2,c=3' should be '1', got {row_a['val_a']}"

print("Problem 5 passed -  str_to_map and element_at verified")

## Problem 6 - named_struct and struct field access

Use `F.named_struct` to create a struct column with explicit field names. Unlike `F.struct` which borrows column names, `F.named_struct` lets you specify each field name as a literal.

From `df_transactions`, build:
```python
F.named_struct(
    F.lit("id"),    F.col("transactionID"),
    F.lit("price"), F.col("totalPrice")
).alias("transaction_summary")
```
Then access the struct fields using `.getField("id")` and `.getField("price")` into separate columns.

**Expected columns:** `transactionID`, `totalPrice`, `transaction_summary`, `summary_id`, `summary_price`  
Store result as `result_6`.

In [ ]:
result_6 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 6 --
assert result_6 is not None, "result_6 is None"

cols_6 = [c.lower() for c in result_6.columns]
for col_name in ["transactionid", "totalprice", "transaction_summary", "summary_id", "summary_price"]:
    assert col_name in cols_6, f"Missing column: {col_name}"

struct_type = result_6.schema["transaction_summary"].dataType
assert isinstance(struct_type, T.StructType), "transaction_summary must be StructType"
field_names = [f.name for f in struct_type.fields]
assert "id" in field_names, "named_struct must have field 'id'"
assert "price" in field_names, "named_struct must have field 'price'"

mismatch_id = result_6.filter(F.col("summary_id") != F.col("transactionID"))
assert mismatch_id.count() == 0, "summary_id must equal transactionID"

mismatch_price = result_6.filter(F.col("summary_price") != F.col("totalPrice"))
assert mismatch_price.count() == 0, "summary_price must equal totalPrice"

print("Problem 6 passed -  named_struct and field access verified")

## Problem 7 - json_tuple

Create the following DataFrame of JSON strings:
```python
spark.createDataFrame([
    ('{"name":"Alice","score":95,"city":"NYC"}',),
    ('{"name":"Bob","score":87,"city":"LA"}',)
], ["json_str"])
```
Use `F.json_tuple(col, "name", "score", "city")` to extract all three fields in one call. This returns columns named `c0`, `c1`, `c2`.  
Also extract `name` using `F.get_json_object(col, "$.name")` into a column called `name_gjo` to compare the two approaches.

**Expected columns:** `json_str`, `c0`, `c1`, `c2`, `name_gjo`  
Store result as `result_7`.

In [ ]:
result_7 = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 7 --
assert result_7 is not None, "result_7 is None"

cols_7 = [c.lower() for c in result_7.columns]
for col_name in ["json_str", "c0", "c1", "c2", "name_gjo"]:
    assert col_name in cols_7, f"Missing column: {col_name}"

rows = result_7.collect()
names = {r["c0"] for r in rows}
assert "Alice" in names and "Bob" in names, "c0 should contain the name values"

scores = {r["c1"] for r in rows}
assert "95" in scores or 95 in scores, "c1 should contain the score values"

# c0 and name_gjo should match
mismatch = result_7.filter(F.col("c0") != F.col("name_gjo"))
assert mismatch.count() == 0, "c0 (json_tuple name) should equal name_gjo (get_json_object name)"

print("Problem 7 passed -  json_tuple and get_json_object verified")

## Problem 8 - sort_array, array_distinct, array_union on real data

From `df_transactions`, use `collect_list` to aggregate all products (with duplicates) per franchise.  
Then apply:
- `F.sort_array(col)` - sort the product list alphabetically
- `F.array_distinct(col)` - remove duplicate product names
- `F.array_union(arr1, arr2)` - take two arbitrary franchises' unique product lists and combine them without duplicates

Add `product_count_before` (size of raw collected list) and `product_count_after` (size after distinct).  
For `array_union`, join the per-franchise result to itself (e.g. self-join on a lagged franchiseID or cross-join two specific franchises) to produce a `union_products` column.

**Expected columns:** `franchiseID`, `all_products`, `sorted_products`, `unique_products`, `product_count_before`, `product_count_after`  
Store the per-franchise result as `result_8` and the union demo as `result_8_union`.

In [ ]:
result_8 = None
result_8_union = None
# YOUR CODE HERE

In [ ]:
# -- Tests for Problem 8 --
assert result_8 is not None, "result_8 is None"
assert result_8_union is not None, "result_8_union is None"

cols_8 = [c.lower() for c in result_8.columns]
for col_name in ["franchiseid", "all_products", "sorted_products", "unique_products",
                 "product_count_before", "product_count_after"]:
    assert col_name in cols_8, f"Missing column: {col_name}"

cnt = result_8.count()
assert cnt > 0, "Got 0 rows"

# product_count_after <= product_count_before (distinct reduces or equals)
violating = result_8.filter(F.col("product_count_after") > F.col("product_count_before"))
assert violating.count() == 0, "product_count_after must be <= product_count_before"

# sorted_products should be in ascending order - verify first element <= last element
order_check = result_8.filter(
    F.size(F.col("sorted_products")) > 1
).filter(
    F.element_at(F.col("sorted_products"), 1) > F.element_at(F.col("sorted_products"), -1)
)
assert order_check.count() == 0, "sorted_products first element should be <= last element"

# result_8_union must have an array column
union_array_cols = [
    f.name for f in result_8_union.schema.fields
    if isinstance(f.dataType, T.ArrayType)
]
assert len(union_array_cols) > 0, "result_8_union must have at least one ArrayType column"

print(f"Problem 8 passed -  sort_array, array_distinct, array_union verified ({cnt} franchises)")